In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
from scipy.stats import ks_2samp, anderson_ksamp

import warnings
import tqdm

## Reading samples

In [14]:
harps = pd.read_csv('harps_sample.csv')
apogee = pd.read_csv('apogee_sample.csv')
galah = pd.read_csv('galah_sample.csv')
gr8 = pd.read_csv('gr8_not_in_harps_sample.csv')

apogee = apogee[apogee.parallax > 0.]
galah = galah[(galah.parallax > 0.) ]

apogee['rel_parallax'] = apogee.parallax_error / apogee.parallax
apogee = apogee[apogee.rel_parallax < 0.1]

galah['rel_parallax'] = galah.parallax_error / galah.parallax
galah = galah[galah.rel_parallax < 0.1]


harps['rel_parallax'] = harps.parallax_error / harps.parallax
harps = harps[harps.rel_parallax < 0.1]

# Seelcting only giants
apogee = apogee[apogee.logg < 3]

# Selecting only dwarfs
galah = galah[galah.logg > 3]


apogee.isnull().sum() # 196 null value for P_gal
apogee.dropna(inplace = True)

galah.isnull().sum() # 1 null value for P_gal
galah.dropna(inplace = True)

#harps.isnull().sum() # No null values


galah = galah[(galah.zmax < 10) ]

apogee = apogee[(apogee.zmax < 10) ]
apogee.shape


print (f'harps contains {len(harps)} stars')
print (f'apogee contains {len(apogee)} stars')    
print (f'galah contains {len(galah)} stars')
print (f'gr8 contains {len(gr8)} stars')

harps contains 798 stars
apogee contains 45727 stars
galah contains 69449 stars
gr8 contains 1430 stars


In [15]:
columns_for_abundances = ['U_lsr', 'V_lsr', 'W_lsr', 'P_gal', 'rapo', 'rperi', 'eccentricity', 'zmax', 'fe_h', 'o_h', 'mg_h', 'si_h', 'c_h']

apogee = apogee[columns_for_abundances]
harps = harps[columns_for_abundances]

In [16]:
PRETTY_LABELS = {
    "U_lsr":        r"$U_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "V_lsr":        r"$V_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "W_lsr":        r"$W_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UV_lsr":       r"$UV_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UW_lsr":       r"$UW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "VW_lsr":       r"$VW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "UVW_lsr":      r"$UVW_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "abs_U_lsr":    r"$|U|_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "abs_W_lsr":    r"$|W|_{\mathrm{LSR}}\,[\mathrm{km\,s^{-1}}]$",
    "P_gal":        r"$P_{\mathrm{gal}}\,[\%]$",
    "rapo":         r"$R_{\mathrm{apo}}\,[\mathrm{kpc}]$",
    "rperi":        r"$R_{\mathrm{peri}}\,[\mathrm{kpc}]$",
    "zmax":         r"$Z_{\max}\,[\mathrm{kpc}]$",
    "eccentricity": r"$e$",
    "R_mean":       r"$R_{\mathrm{mean}}\,[\mathrm{kpc}]$",
    "o_h":          r"$[\mathrm{O/H}]$",
    "mg_h":         r"$[\mathrm{Mg/H}]$",
    "si_h":         r"$[\mathrm{Si/H}]$",
    "c_h":          r"$[\mathrm{C/H}]$",
}

In [17]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

color_list_2 = ["#1f77b4", "#ff7f0e"]  # APOGEE, HARPS

def plot_distribution_duo(
    df1, df2, feature,
    labels=("APOGEE", "HARPS"),
    pretty_labels=None,               # <- pass PRETTY_LABELS here
    source_col="source",              # <- fixed grouping col
    palette=color_list_2,
    bins=30,
    clip_quantiles=(0.0, 1.0),        # e.g. (0.005, 0.995) if you want trimming
    kde=True,
    figsize=(7, 8),
    savepath=None,
    dpi=300,
    show=True
):
    """
    Two-panel comparison plot:
      (top) overlaid histograms (probability)
      (bottom) boxplot by dataset

    Uses pretty_labels[feature] if provided for axis labels.
    """

    # --- build combined dataset with labels
    df1_copy = df1.copy()
    df2_copy = df2.copy()
    df1_copy[source_col] = labels[0]
    df2_copy[source_col] = labels[1]
    data_df = pd.concat([df1_copy, df2_copy], ignore_index=True)

    # --- clean numeric feature
    data_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_df = data_df[pd.to_numeric(data_df[feature], errors="coerce").notna()]
    data_df[feature] = data_df[feature].astype(float)

    # --- pretty axis label
    if pretty_labels is None:
        pretty_labels = {}
    xlab = pretty_labels.get(feature, feature)

    # --- plot
    f, axes = plt.subplots(2, 1, figsize=figsize)

    # histogram per dataset
    for i, s in enumerate(data_df[source_col].unique()):
        selection = data_df.loc[data_df[source_col] == s, feature].dropna()
        if len(selection) < 10:
            continue

        qlo, qhi = np.quantile(selection, clip_quantiles)
        selection = selection[(selection >= qlo) & (selection <= qhi)]

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=FutureWarning)
            sns.histplot(
                selection,
                bins=bins,
                color=palette[i],
                ax=axes[0],
                label=s,
                kde=kde,
                stat="probability",
                element="step",
                fill=True,
                alpha=0.4,
            )

    # boxplot
    sns.boxplot(
        x=source_col, y=feature,
        data=data_df,
        palette=palette,
        ax=axes[1]
    )

    # labels
    axes[0].set_xlabel(xlab, fontsize=14)
    axes[0].set_ylabel("Probability", fontsize=14)
    axes[1].set_xlabel("")
    axes[1].set_ylabel(xlab, fontsize=14)

    # ticks
    for ax in axes:
        ax.tick_params(axis="both", which="major", labelsize=12, length=6, width=1.5)

    axes[0].legend(title="")
    plt.tight_layout()

    if savepath is not None:
        os.makedirs(os.path.dirname(savepath), exist_ok=True)
        plt.savefig(savepath, dpi=dpi)

    if show:
        plt.show()
    else:
        plt.close()

In [18]:
def plot_all_feature_distributions(
    df1, df2,
    features,
    labels=("APOGEE", "HARPS"),
    pretty_labels=None,
    outdir="./figures/feature_distributions",
    **kwargs
):
    """
    Makes one figure per feature and saves them to outdir.
    Any kwargs are passed to plot_distribution_duo.
    """
    os.makedirs(outdir, exist_ok=True)

    for feat in features:
        savepath = os.path.join(outdir, f"dist_{feat}.png")
        plot_distribution_duo(
            df1, df2,
            feature=feat,
            labels=labels,
            pretty_labels=pretty_labels,
            savepath=savepath,
            **kwargs
        )

In [19]:
FEATURES = apogee.columns

plot_all_feature_distributions(
    apogee, harps,
    features=FEATURES,
    labels=("APOGEE", "HARPS"),
    pretty_labels=PRETTY_LABELS,
    outdir="./figures/feature_distributions",
    bins=30,
    clip_quantiles=(0.0, 1.0),   # set to (0.005, 0.995) if you want to suppress extreme tails
    kde=True,
    show=False                   # set True if you want them displayed interactively
)

/tmp/ipykernel_88621/3608817409.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_88621/3608817409.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_88621/3608817409.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_88621/3608817409.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/tmp/ipykernel_88621/3608817409.py:77: FutureWarning